# Task 1 16×16 spatial autoencoder experiment
Push the new files to GitHub first. Run setup, smoke check, then full training in order. Enable a GPU runtime. This is a fresh model, not a continuation of the vector checkpoint. Settings are provisional, not Optuna-selected. Do not use Run all: the optional resume cell is for interrupted training only.


In [ ]:
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# Before starting a new experiment, fetch the implementation you pushed.
# When resuming, keep the source revision used by your checkpoint.
subprocess.run(['git', 'pull', '--ff-only'], check=True)
assert (repo / 'training/train_task1_spatial16.py').is_file(), 'Push the new files first'
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


If Colab requests a session restart after installation, restart and rerun setup. Keep the original source revision when resuming; provenance checks reject changed training sources.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
os.chdir(repo)
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)
output = Path('/content/drive/MyDrive/GenAI_A1/task1-spatial16')


## Smoke check
This only verifies execution. Its 16-image, one-epoch metrics do not measure final quality. Skip if already completed successfully.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.train_task1_spatial16', '--device', 'cuda',
    '--epochs', '1', '--train-limit', '16', '--val-images', '1',
    '--output-dir', str(output.parent / 'task1-spatial16-smoke')], check=True)


## Fresh five-epoch pilot
20-epoch target, all 2,944 training images and 7,360 fixed validation cases. The pilot pauses after epoch 5 while keeping a 20-epoch target for compatible resume. The 4.5-hour soft budget allows saving time; Colab runtime duration is not guaranteed. Corruptions are generated during training.


In [ ]:
subprocess.run([sys.executable, '-m', 'training.train_task1_spatial16', '--device', 'cuda',
    '--max-hours', '4.5', '--stop-after-epoch', '5', '--output-dir', str(output)], check=True)


## Optional resume — after reviewing the pilot
After reviewing the five-epoch pilot, restore setup if needed and run this instead of the fresh-run cell to continue toward epoch 20. Keep the same config and source files. Never pass an old vector-model checkpoint.


In [ ]:
# Uncomment only to resume an interrupted spatial run.
# subprocess.run([sys.executable, '-m', 'training.train_task1_spatial16', '--device', 'cuda',
#     '--max-hours', '4.5', '--output-dir', str(output),
#     '--resume', str(output / 'last.pt')], check=True)


## Evaluate best checkpoint
Run after the pilot pauses or full training finishes. Use a fresh analysis directory for each evaluation. Compare with the dropout-zero vector baseline on the same validation cases. Official test images remain unused. Analysis refuses to overwrite a nonempty folder.

For evaluation after the later full run, change the analysis folder in both cells below to `validation_analysis_full` to preserve the pilot report.


In [ ]:
history = __import__('json').loads((output / 'history.json').read_text())
assert history, 'Complete at least one epoch before evaluating'
subprocess.run([sys.executable, '-m', 'evaluation.evaluate_task1_spatial16', '--device', 'cuda',
    '--checkpoint', str(output / 'best.pt'), '--output-dir', str(output / 'validation_analysis')], check=True)


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((output / 'validation_analysis/README.md').read_text()))
display(Image(filename=str(output / 'validation_analysis/representative_grid.png')))


Back up the whole `MyDrive/GenAI_A1/task1-spatial16` folder. Share `history.json`, `validation_analysis/summary.json`, and `validation_analysis/representative_grid.png` for review. `preview.png` is from the latest epoch; the analysis uses the best checkpoint.
